# Semantic-Adaptive 3DGS：L4/T4 多视角高细节训练

这个 Notebook 使用 **NeRF Synthetic Lego** 的 80 个训练视角与 10 个验证视角，在 512×512 分辨率下运行官方 CUDA Gaussian Rasterizer，并完成：SAM+CLIP 预处理 → 重要区域加权 RGB 3DGS → 语义蒸馏 → 文本搜索 → 非破坏性删除 → Gaussian Atlas 网页导出。重要区域词汇已由项目助手预生成，不调用外部 LLM API。

运行前在 `Runtime / 运行时 → Change runtime type / 更改运行时类型` 中选择 **L4 GPU（推荐）** 或 T4 GPU。默认进行 40,000 次 RGB 与 8,000 次语义迭代；保留官方自适应分裂到 22,000 轮，并只清理真正巨大的尺度离群点。L4 更快，T4 更节省额度；无需 A100/H100。

In [ ]:
import os, platform, subprocess, sys, time
import torch

assert torch.cuda.is_available(), '请先把 Colab 运行时改为 L4 或 T4 GPU，然后重新运行。'
GPU_NAME = torch.cuda.get_device_name(0)
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__, 'CUDA:', torch.version.cuda)
print('GPU:', GPU_NAME)
if not any(name in GPU_NAME.upper() for name in ('L4', 'T4')):
    raise RuntimeError(f'当前是 {GPU_NAME}。请选择 L4（推荐）或 T4 GPU。')
subprocess.run(['nvidia-smi'], check=True)


In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/Xuyw041006-arch/gaussian-splatting.git'
REPO = Path('/content/gaussian-splatting')
RUN_ROOT = Path('/content/semantic_3dgs_multiview_detail')
SCENE = RUN_ROOT / 'lego_multiview80_512'
MODEL = RUN_ROOT / 'model'
EDITED_MODEL = RUN_ROOT / 'model_without_query'
SAM_CHECKPOINT = Path('/content/checkpoints/sam_vit_h_4b8939.pth')
WEB_VIEWER_URL = 'https://gaussian-atlas-xyw.xuyw041006.chatgpt.site'
TRAIN_VIEWS = 80
TEST_VIEWS = 10
IMAGE_SIZE = 512
INITIAL_POINTS = 100000
RGB_ITERATIONS = 40000
SEMANTIC_ITERATIONS = 8000
IMPORTANT_OBJECTS = [
    'yellow Lego bulldozer',
    'yellow vehicle body',
    'black excavator arm',
    'gray caterpillar track',
    'red wheel hub',
]
print('精细训练目录:', RUN_ROOT)
print(f'配置: {TRAIN_VIEWS} 训练视角 + {TEST_VIEWS} 验证视角 / {IMAGE_SIZE}px / RGB {RGB_ITERATIONS} / Semantic {SEMANTIC_ITERATIONS}')


## 1. 克隆代码并编译真实 CUDA 扩展

第一次运行通常最慢，因为这里会现场编译官方 rasterizer；这一步成功才算真正具备 3DGS 运行环境。

In [ ]:
def run(command, cwd=None):
    printable = ' '.join(map(str, command))
    print(f'\n$ {printable}')
    started = time.time()
    subprocess.run([str(item) for item in command], cwd=cwd, check=True)
    print(f'完成，用时 {(time.time() - started) / 60:.1f} 分钟')

if not REPO.exists():
    run(['git', 'clone', '--recursive', REPO_URL, REPO])
else:
    run(['git', '-C', REPO, 'pull', '--ff-only'])
    run(['git', '-C', REPO, 'submodule', 'update', '--init', '--recursive'])

os.environ['MAX_JOBS'] = '4'
run([sys.executable, '-m', 'pip', 'install', '-q', 'ninja', 'plyfile', 'huggingface_hub'])
for package in ('diff-gaussian-rasterization', 'simple-knn', 'fused-ssim'):
    run([sys.executable, '-m', 'pip', 'install', '-q', '--no-build-isolation', REPO / 'submodules' / package])
run([sys.executable, '-m', 'pip', 'install', '-q', '-r', REPO / 'requirements-semantic.txt'])
run([sys.executable, REPO / 'scripts/preflight.py'], cwd=REPO)


## 2. 下载标准 Lego 数据并构造 80 视角精细场景

均匀下载 80 张训练图和 10 张独立验证图，并缩放到 512 像素。相较原来的 8×256 配置，这会显著增强遮挡区域覆盖、轮廓连续性和细小结构；数据来自公开的 NeRF Synthetic 镜像。

In [ ]:
import json, shutil
import numpy as np
from huggingface_hub import hf_hub_download

DATA_REPO = 'phuckstnk63/nerf-synthetic'
RAW_ROOT = Path('/content/nerf_synthetic_raw')
FULL_SCENE = Path('/content/nerf_synthetic_stage/lego')
if FULL_SCENE.exists():
    shutil.rmtree(FULL_SCENE)
(FULL_SCENE / 'train').mkdir(parents=True, exist_ok=True)
json_name = 'nerf_synthetic/lego/transforms_train.json'
json_path = Path(hf_hub_download(DATA_REPO, json_name, repo_type='dataset', local_dir=RAW_ROOT))
transforms = json.loads(json_path.read_text())
indices = np.linspace(0, len(transforms['frames']) - 1, TRAIN_VIEWS + TEST_VIEWS, dtype=int)
held_out_positions = set(np.linspace(1, len(indices) - 2, TEST_VIEWS, dtype=int).tolist())
train_frames, test_frames = [], []
for position, index in enumerate(indices):
    frame = transforms['frames'][int(index)]
    relative = frame['file_path'].removeprefix('./') + '.png'
    downloaded = Path(hf_hub_download(DATA_REPO, f'nerf_synthetic/lego/{relative}', repo_type='dataset', local_dir=RAW_ROOT))
    destination = FULL_SCENE / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(downloaded, destination)
    (test_frames if position in held_out_positions else train_frames).append(frame)
for split, frames in (('train', train_frames), ('test', test_frames)):
    payload = {'camera_angle_x': transforms['camera_angle_x'], 'frames': frames}
    (FULL_SCENE / f'transforms_{split}.json').write_text(json.dumps(payload, indent=2))
assert len(train_frames) == TRAIN_VIEWS and len(test_frames) == TEST_VIEWS

if SCENE.exists():
    shutil.rmtree(SCENE)
run([sys.executable, REPO / 'scripts/prepare_nerf_sparse_scene.py',
     '--source', FULL_SCENE, '--output', SCENE, '--train_views', str(TRAIN_VIEWS),
     '--test_views', str(TEST_VIEWS), '--size', str(IMAGE_SIZE),
     '--points', str(INITIAL_POINTS)], cwd=REPO)

from PIL import Image
from IPython.display import display
display(Image.open(sorted((SCENE / 'images').glob('*.png'))[0]).resize((256, 256)))
IMPORTANT_JSON = SCENE / 'important_objects.generated.json'
prompt_map = {path.name: IMPORTANT_OBJECTS for path in sorted((SCENE / 'images').glob('*.png'))}
IMPORTANT_JSON.write_text(json.dumps(prompt_map, ensure_ascii=False, indent=2))
print('助手预生成的重要区域词汇:', json.dumps(IMPORTANT_OBJECTS, ensure_ascii=False))


## 3. SAM + OpenCLIP 语义与重要区域

这里下载精度更高的 SAM ViT-H 权重，并读取上一单元生成的 `important_objects.generated.json`。语义编码升级为 OpenCLIP ViT-H/14，SAM 掩码会按 predicted-IoU 与 stability 自动加权。它与将来 LLM API 的输出格式一致，但本次不产生 API 调用或费用。

In [ ]:
import urllib.request

SAM_CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
if not SAM_CHECKPOINT.exists():
    urllib.request.urlretrieve(
        'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth',
        SAM_CHECKPOINT,
    )
run([sys.executable, REPO / 'preprocess_semantics.py',
     '--scene', SCENE, '--sam_checkpoint', SAM_CHECKPOINT, '--sam_model', 'vit_h',
     '--clip_model', 'ViT-H-14', '--clip_pretrained', 'laion2b_s32b_b79k',
     '--important_json', IMPORTANT_JSON,
     '--feature_dim', '24', '--feature_width', '512', '--min_mask_area', '64',
     '--max_masks', '192', '--points_per_side', '32', '--batch_size', '16',
     '--importance_topk', '3', '--device', 'cuda'], cwd=REPO)
assert (SCENE / 'semantic_meta.npz').is_file()
assert len(list((SCENE / 'semantic_maps').glob('*.npz'))) == TRAIN_VIEWS
print((SCENE / 'semantic_summary.json').read_text())


## 4. 重要区域加权 RGB 3DGS（真实 CUDA 训练）

In [ ]:
if MODEL.exists():
    shutil.rmtree(MODEL)
run([sys.executable, REPO / 'train.py', '-s', SCENE, '-m', MODEL, '--eval',
     '--antialiasing', '--iterations', str(RGB_ITERATIONS),
     '--save_iterations', '7000', '22000', str(RGB_ITERATIONS),
     '--test_iterations', '7000', '22000', str(RGB_ITERATIONS),
     '--importance_mask_dir', SCENE / 'importance_masks',
     '--foreground_weight', '3.0', '--background_weight', '0.75',
     '--densify_from_iter', '500', '--densify_until_iter', '22000',
     '--densify_grad_threshold', '0.0001',
     '--densification_interval', '100', '--opacity_reset_interval', '3000',
     '--disable_viewer', '--quiet'], cwd=REPO)
RGB_PLY = MODEL / 'point_cloud' / f'iteration_{RGB_ITERATIONS}' / 'point_cloud.ply'
assert RGB_PLY.is_file(), RGB_PLY
print('RGB 3DGS 输出:', RGB_PLY, RGB_PLY.stat().st_size, 'bytes')


## 5. 清理巨大高斯与空间离群点

NeRF Synthetic Lego 的随机初始点云会留下少量巨大或远离主体的高斯。高细节预设把透明度阈值放宽到 0.005，保留官方分裂产生的小高斯；尺度与半径阈值只负责去掉明显离群点。以下参数适用于本 Lego 演示场景，自建场景应根据尺度重新调节。原始 PLY 会备份。

In [ ]:
RAW_RGB_PLY = RGB_PLY.with_name('point_cloud.unpruned.ply')
PRUNED_RGB_PLY = RGB_PLY.with_name('point_cloud.pruned.ply')
shutil.copy2(RGB_PLY, RAW_RGB_PLY)
run([sys.executable, REPO / 'scripts/prune_gaussians.py',
     '--input', RGB_PLY, '--output', PRUNED_RGB_PLY,
     '--max_scale', '0.014', '--min_opacity', '0.005',
     '--max_radius', '1.30'], cwd=REPO)
shutil.move(PRUNED_RGB_PLY, RGB_PLY)
from plyfile import PlyData
print('清理后高斯数:', len(PlyData.read(RGB_PLY)['vertex'].data))
print('原始模型备份:', RAW_RGB_PLY)


## 6. 把二维语义蒸馏到清理后的三维高斯

In [ ]:
run([sys.executable, REPO / 'train_semantics.py', '-m', MODEL,
     '--iteration', str(RGB_ITERATIONS),
     '--semantic_iterations', str(SEMANTIC_ITERATIONS),
     '--semantic_lr', '0.005', '--spatial_weight', '0.02',
     '--spatial_k', '8', '--spatial_samples', '4096',
     '--save_every', '0', '--quiet'], cwd=REPO)
SEMANTIC_FILE = MODEL / 'semantic' / f'iteration_{RGB_ITERATIONS}' / 'semantic_features.pt'
assert SEMANTIC_FILE.is_file(), SEMANTIC_FILE
print('三维语义输出:', SEMANTIC_FILE, SEMANTIC_FILE.stat().st_size, 'bytes')


## 7. 文本搜索和删除验证

40,000/8,000 次训练完成后，用宽松阈值取最高分的 5,000 个高斯做确定性的搜索与删除验证。实际使用时应根据验证视角调节语义阈值；删除始终写入新模型，原模型保持不变。

In [ ]:
SELECTION = RUN_ROOT / 'yellow_lego_selection.npz'
QUERY_JSON = RUN_ROOT / 'yellow_lego_result.json'
SELECTED_PLY = RUN_ROOT / 'yellow_lego_selected.ply'
run([sys.executable, REPO / 'semantic_query.py', '--model', MODEL,
     '--text', 'yellow Lego bulldozer', '--iteration', str(RGB_ITERATIONS),
     '--threshold', '-1.0', '--top_k', '5000', '--output', SELECTION,
     '--json', QUERY_JSON, '--export_selected', SELECTED_PLY, '--device', 'cuda'], cwd=REPO)
if EDITED_MODEL.exists():
    shutil.rmtree(EDITED_MODEL)
run([sys.executable, REPO / 'semantic_edit.py', '--model', MODEL,
     '--selection', SELECTION, '--output_model', EDITED_MODEL, '--action', 'remove'], cwd=REPO)

from plyfile import PlyData
original_count = len(PlyData.read(RGB_PLY)['vertex'].data)
selected_count = len(np.load(SELECTION)['indices'])
edited_ply = EDITED_MODEL / 'point_cloud' / f'iteration_{RGB_ITERATIONS}' / 'point_cloud.ply'
edited_count = len(PlyData.read(edited_ply)['vertex'].data)
assert selected_count > 0
assert edited_count == original_count - selected_count
assert RGB_PLY.is_file(), '源模型不应被删除操作修改'
result = json.loads(QUERY_JSON.read_text())
print(json.dumps(result, ensure_ascii=False, indent=2))
print(f'删除验证通过：{original_count} - {selected_count} = {edited_count}')


## 8. 导出到独立的 Gaussian Atlas 网页

直接复用上面完成的 40,000 次 RGB 与 8,000 次语义模型，不再重复训练。这里导出网页需要的 `point_cloud.ply` 与 `semantic_objects.json`。Gaussian Atlas 支持直接拖动旋转、滚轮缩放、单击选择、文本搜索和可恢复删除；Colab 只负责训练与导出。

In [ ]:
DISPLAY_ITERATIONS = RGB_ITERATIONS
DISPLAY_MODEL = MODEL
DISPLAY_PLY = DISPLAY_MODEL / 'point_cloud' / f'iteration_{DISPLAY_ITERATIONS}' / 'point_cloud.ply'
assert DISPLAY_PLY.is_file(), DISPLAY_PLY
DISPLAY_SEMANTIC = DISPLAY_MODEL / 'semantic' / f'iteration_{DISPLAY_ITERATIONS}' / 'semantic_features.pt'
assert DISPLAY_SEMANTIC.is_file(), DISPLAY_SEMANTIC
print('✓ 复用精细模型进行网页导出：', DISPLAY_PLY)


In [ ]:
WEB_EXPORT = DISPLAY_MODEL / 'web_export' / f'iteration_{DISPLAY_ITERATIONS}'
run([sys.executable, REPO / 'export_web_bundle.py',
     '--model', DISPLAY_MODEL, '--iteration', str(DISPLAY_ITERATIONS),
     '--labels', ','.join(IMPORTANT_OBJECTS), '--threshold', '0.20',
     '--top_k', '30000', '--output_dir', WEB_EXPORT], cwd=REPO)
WEB_ARCHIVE = shutil.make_archive('/content/gaussian_atlas_web_bundle', 'zip', WEB_EXPORT)
print('网页地址:', WEB_VIEWER_URL)
print('网页导入包:', WEB_ARCHIVE)
print('使用方法：解压 ZIP，在网页依次导入 point_cloud.ply 和 semantic_objects.json')
from IPython.display import Markdown, display
display(Markdown(f'### [打开 Gaussian Atlas 交互网页]({WEB_VIEWER_URL})'))


In [ ]:
archive = shutil.make_archive('/content/semantic_3dgs_multiview_hq_results', 'zip', RUN_ROOT)
print('✅ L4/T4 多视角高质量训练与全链路验证通过')
print('完整结果压缩包:', archive)
print('网页导入包:', WEB_ARCHIVE)
print('需要下载时运行：from google.colab import files; files.download(archive)')
